# Traditional Machine Learning — full

This notebook contains the direct TF-IDF implementation for Logistic Regression and Linear SVM.

Historical results are in results/full_data. A new run writes to reproduced_runs.

Approximate historical CPU time: Logistic Regression 1 minute; Linear SVM 2 minutes.

Settings:

- train / validation / test: 1,424,657 / 178,082 / 178,083
- TF-IDF: lowercase, sublinear TF, 1–2 grams, min_df 2, 50,000 features, float32
- Logistic Regression: C 1.0, liblinear, max_iter 1000, balanced classes
- Linear SVM: C 1.0, max_iter 5000, balanced classes, dual auto
- seed: 42

In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = next(
    path for path in (Path.cwd(), Path.cwd().parent)
    if (path / "data/raw/train.csv").exists()
)
REPRODUCED = ROOT / "reproduced_runs"


EXPERIMENT = "full"
TRAIN_PATH = ROOT / "data/splits/full/train.csv"
VALIDATION_PATH = ROOT / "data/splits/full/validation.csv"
TEST_PATH = ROOT / "data/splits/full/test.csv"
OUTPUT_DIRS = {
    "logistic_regression": REPRODUCED / EXPERIMENT / "logistic_regression",
    "linear_svm": REPRODUCED / EXPERIMENT / "linear_svm",
}
TFIDF_SETTINGS = {
    "lowercase": True,
    "sublinear_tf": True,
    "ngram_range": (1, 2),
    "min_df": 2,
    "max_features": 50000,
    "dtype": np.float32,
}
MODEL_SETTINGS = {
    "logistic_regression": {"C": 1.0, "solver": "liblinear", "max_iter": 1000, "class_weight": "balanced", "random_state": 42},
    "linear_svm": {"C": 1.0, "max_iter": 5000, "class_weight": "balanced", "dual": "auto", "random_state": 42},
}

## Load and check the data

In [4]:
train = pd.read_csv(TRAIN_PATH)
validation = pd.read_csv(VALIDATION_PATH)
test = pd.read_csv(TEST_PATH)
assert len(train) == 1424657
assert len(validation) == 178082
assert len(test) == 178083
assert train["target"].ge(0.5).astype("int8").equals(train["label"].astype("int8"))
assert validation["target"].ge(0.5).astype("int8").equals(validation["label"].astype("int8"))
assert test["target"].ge(0.5).astype("int8").equals(test["label"].astype("int8"))
assert train["id"].is_unique and validation["id"].is_unique and test["id"].is_unique
assert not (set(train["id"]) & set(validation["id"]))
assert not (set(train["id"]) & set(test["id"]))
assert not (set(validation["id"]) & set(test["id"]))

## Core TF-IDF and model training code

In [6]:
RUN_TRAINING = True

if RUN_TRAINING:
    import json
    import time
    import joblib
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import f1_score
    from sklearn.svm import LinearSVC

    if any(path.exists() for path in OUTPUT_DIRS.values()):
        raise FileExistsError("A reproduced output already exists.")

    vectorizer = TfidfVectorizer(**TFIDF_SETTINGS)
    train_matrix = vectorizer.fit_transform(train["comment_text"])
    validation_matrix = vectorizer.transform(validation["comment_text"])
    test_matrix = vectorizer.transform(test["comment_text"])
    models = {
        "logistic_regression": LogisticRegression(**MODEL_SETTINGS["logistic_regression"]),
        "linear_svm": LinearSVC(**MODEL_SETTINGS["linear_svm"]),
    }
    for name, model in models.items():
        started = time.perf_counter()
        model.fit(train_matrix, train["label"])
        validation_prediction = model.predict(validation_matrix)
        print(name, "validation toxic F1:", f1_score(validation["label"], validation_prediction, zero_division=0))
        output_dir = OUTPUT_DIRS[name]
        output_dir.mkdir(parents=True, exist_ok=False)
        output = test[["id", "comment_text", "target", "label"]].copy()
        output["predicted_label"] = model.predict(test_matrix)
        if name == "logistic_regression":
            output["toxic_probability"] = model.predict_proba(test_matrix)[:, 1]
        else:
            output["decision_score"] = model.decision_function(test_matrix)
        output.to_csv(output_dir / "predictions.csv", index=False)
        joblib.dump(vectorizer, output_dir / "tfidf_vectorizer.joblib")
        joblib.dump(model, output_dir / "model.joblib")
        tfidf_for_log = dict(TFIDF_SETTINGS)
        tfidf_for_log["dtype"] = "float32"
        (output_dir / "settings.json").write_text(json.dumps({
            "tfidf": tfidf_for_log,
            "model": MODEL_SETTINGS[name],
            "seed": 42,
            "training_seconds": time.perf_counter() - started,
        }, indent=2))